#### Running using run.py

In [ ]:
from run import run_experiment
import run as experiment_runner
import csv
import json
import time
from datetime import datetime
from pathlib import Path
from functools import wraps
import torch
import yaml

# Timing wrappers observe the existing runner; no datasets or training settings change.
# GPU synchronization makes elapsed times include completed GPU work.
TIMING_DIR = Path("runs/benchmarks/notebook2_timings") / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
TIMING_DIR.mkdir(parents=True, exist_ok=False)
timing_results = []

def timed_experiment(config_path):
    config = yaml.safe_load(Path(config_path).read_text())
    training = config["training"]
    epochs = training["epochs"]
    every = training.get("validate_every", 5)
    validation_count = sum(e % every == 0 or e == epochs for e in range(1, epochs + 1))
    evaluation_splits = ["validation"] * validation_count + [
        split for split in ("test", "bolivia") if config["dataset"].get(f"{split}_csv")]
    result = {
        "config_path": str(config_path), "modality": "+".join(config["dataset"]["modalities"]),
        "num_workers": training.get("num_workers", 0), "epochs": epochs,
        "training_batch_size": training["batch_size"],
        "evaluation_batch_size": config["evaluation"].get("batch_size", 1),
        "tile_batch_size": config["evaluation"].get("tile_batch_size", 1),
        "status": "running", "phases": [],
    }
    originals = {name: getattr(experiment_runner, name) for name in
                 ("train_one_epoch", "evaluate", "save_checkpoint")}
    counters = {name: 0 for name in originals}
    def synchronize():
        if torch.cuda.is_available():
            torch.cuda.synchronize()
    def instrument(name, function):
        @wraps(function)
        def measured(*args, **kwargs):
            counters[name] += 1
            index = counters[name]
            label = (f"training epoch {index}" if name == "train_one_epoch" else
                     evaluation_splits[index - 1] if name == "evaluate" and index <= len(evaluation_splits) else
                     name)
            synchronize()
            start = time.perf_counter()
            completed = False
            try:
                value = function(*args, **kwargs)
                completed = True
                return value
            finally:
                synchronize()
                elapsed = time.perf_counter() - start
                row = {"phase": name, "label": label, "seconds": elapsed, "completed": completed}
                if name in ("train_one_epoch", "evaluate"):
                    row["samples"] = len(args[1].dataset)
                    row["samples_per_second"] = row["samples"] / elapsed
                result["phases"].append(row)
                print(f"[Timing] {label}: {elapsed:.2f}s", flush=True)
        return measured
    for name, function in originals.items():
        setattr(experiment_runner, name, instrument(name, function))
    synchronize()
    start = time.perf_counter()
    try:
        result["run_directory"] = str(run_experiment(config_path))
        result["status"] = "complete"
    except BaseException as error:
        result["status"] = "failed_or_interrupted"
        result["error"] = repr(error)
        raise
    finally:
        # Restore normal runner functions even after an error or interruption.
        for name, function in originals.items():
            setattr(experiment_runner, name, function)
        synchronize()
        result["total_seconds"] = time.perf_counter() - start
        for name, key in [("train_one_epoch", "training_seconds"),
                          ("evaluate", "evaluation_seconds"),
                          ("save_checkpoint", "checkpoint_seconds")]:
            result[key] = sum(row["seconds"] for row in result["phases"] if row["phase"] == name)
        result["other_seconds"] = result["total_seconds"] - sum(
            result[k] for k in ("training_seconds", "evaluation_seconds", "checkpoint_seconds"))
        # Compare only completed runs whose configurations match apart from workers/names/output.
        comparison = json.loads(json.dumps(config))
        comparison["training"].pop("num_workers", None)
        comparison["training"].setdefault("validate_every", 5)
        comparison["experiment"].pop("name", None)
        comparison.pop("output", None)
        result["comparison_config"] = comparison
        baseline = next((r for r in timing_results if r["num_workers"] == 0
                         and r["status"] == "complete" and r["comparison_config"] == comparison), None)
        result["speedup_vs_workers0"] = (baseline["total_seconds"] / result["total_seconds"]
                                         if baseline and result["status"] == "complete" else
                                         1.0 if result["num_workers"] == 0 and result["status"] == "complete" else None)
        timing_results.append(result)
        (TIMING_DIR / "timings.json").write_text(json.dumps(timing_results, indent=2), encoding="utf-8")
        columns = ["config_path", "modality", "num_workers", "epochs", "status", "total_seconds",
                   "training_seconds", "evaluation_seconds", "checkpoint_seconds", "other_seconds", "speedup_vs_workers0"]
        with (TIMING_DIR / "summary.csv").open("w", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=columns, extrasaction="ignore")
            writer.writeheader()
            writer.writerows(timing_results)
        print(f"[Timing] Total: {result['total_seconds']/60:.2f} minutes; reports: {TIMING_DIR}", flush=True)
        if result["speedup_vs_workers0"] is not None:
            print(f"[Timing] Speedup vs workers=0: {result['speedup_vs_workers0']:.2f}x", flush=True)
    return result["run_directory"]


In [ ]:
timed_experiment("configs/croma/croma_test_faster_evaluation_numworkers0.yaml")
timed_experiment("configs/croma/croma_test_faster_evaluation_numworkers2.yaml")
timed_experiment("configs/croma/croma_test_faster_evaluation_numworkers4.yaml")

timed_experiment("configs/croma/croma_s1s2_fcn_all_hand_labels_numworkers0.yaml")
timed_experiment("configs/croma/croma_s1s2_fcn_all_hand_labels_numworkers2.yaml")
timed_experiment("configs/croma/croma_s1s2_fcn_all_hand_labels_numworkers4.yaml")